# 채보 뒤 노트 수 기준값 측정 — GPU 서버 실행 (S15P31A307-321)

채보가 끝난 뒤 노트가 너무 적은 곡을 실패로 끝내는 검사(`configs/chart/transcription.yaml`의 `drum_check.min_notes_per_minute`)의 기준값 92는 이 노트북의 측정으로 정했다. 드럼 있는 곡과 드럼 없는 음원을 서비스와 같은 분리 → 채보로 돌려 1분당 노트 수가 얼마나 떨어져 있는지 측정한다.

- 입력은 MUSDB18-HQ test 50곡의 가운데 90초를 두 가지로 만든 것이다. 원곡 그대로(`musdb_mix`, 드럼 있음)와 같은 구간에서 드럼만 뺀 것(`musdb_nodrum`). 같은 곡이라 드럼 유무만 다르다
- 말소리 묶음(`speech`), 말소리에 곡을 깐 것(`talk_bgm`), 축구 영상·소음(`extra`)도 같이 돌린다
- 모델은 운영 워커와 같은 것을 쓴다. 채보 모델 이름은 `ssaplash/worker.env`의 `SSAPLASH_CHART_MODEL` 한 줄만 읽는다(토큰 등 다른 값은 읽지 않는다)
- 분리 전 음악 여부 검사 분류 모델을 올릴 수 있으면 그 검사 시간도 같이 측정한다. 못 올리면 건너뛴다

**준비물.** 이 파일과 로컬 `AI/scripts/note_count_measure.py`를 JupyterHub `ssaplash/notebooks/`에, 로컬 `AI/data/upload/note_count_inputs.zip`을 `ssaplash/upload/`에 올린다. 코드 폴더 `~/ssaplash/S15P31A307`은 운영 워커가 쓰는 git 체크아웃이라 읽기만 하고 건드리지 않는다.

**순서.** 1번 셀을 한 번 실행하면 백그라운드로 끝까지 돈다(20~40분). 2번 셀은 여러 번 쳐도 되고, 끝나면 요약 표를 출력하고 `ssaplash/results/note_count.zip`으로 묶는다.

**끊기면.** 1번은 nohup이라 브라우저를 닫아도 끝까지 돈다. 커널이 다시 시작됐으면 1번 → 2번을 다시 친다. 1번은 이미 돌고 있거나 끝났으면 새로 시작하지 않는다. 측정이 도중에 멈췄으면 1번을 다시 실행한다. 끝난 파일은 건너뛰고 남은 파일만 돈다.

In [ ]:
# 1. 설정·점검·측정 시작 (한 번만. 이미 돌고 있거나 끝났으면 다시 시작하지 않고 알려 줌)
import getpass
import os
import subprocess
import zipfile

GPU_ID = "6"
HOME = os.path.expanduser("~")
WORK = f"{HOME}/ssaplash"
AI = f"{WORK}/S15P31A307/AI"                      # 운영 워커 코드 폴더. 읽기만 함
SCRIPT = f"{WORK}/notebooks/note_count_measure.py"
INPUT_ZIP = f"{WORK}/upload/note_count_inputs.zip"
INPUT_DIR = f"{WORK}/upload/note_count_inputs"
OUT = f"{WORK}/results/note-count"
LOG = f"{WORK}/results/note_count.log"

# 운영 워커와 같은 채보 모델을 씀. worker.env에서 모델 이름 한 줄만 읽음
chart_model = ""
if os.path.isfile(f"{WORK}/worker.env"):
    for line in open(f"{WORK}/worker.env"):
        line = line.strip().removeprefix("export ")
        if line.startswith("SSAPLASH_CHART_MODEL="):
            chart_model = line.split("=", 1)[1].split("#")[0].strip()

# 운영 워커와 같은 환경: GPU 6번, CPU 0~127번, 스레드 16
# MPLBACKEND: Jupyter 커널의 inline 그림 설정이 넘어가면 분리 모델을 못 올림. 운영 워커처럼 Agg
env = dict(os.environ, CUDA_VISIBLE_DEVICES=GPU_ID, SSAPLASH_CHART_DEVICE="cuda", PYTHONUTF8="1",
           PYTHONUNBUFFERED="1", MPLBACKEND="Agg", OMP_NUM_THREADS="16", MKL_NUM_THREADS="16",
           OPENBLAS_NUM_THREADS="16", NUMEXPR_NUM_THREADS="16")
if chart_model:
    env["SSAPLASH_CHART_MODEL"] = chart_model
else:
    env.pop("SSAPLASH_CHART_MODEL", None)

!nvidia-smi -i $GPU_ID --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv
!cat /proc/loadavg
!cd $AI && git log --oneline -1
print("채보 모델:", chart_model or "(worker.env에 없음 → 설정 파일 기본값)")

assert os.path.isfile(SCRIPT), "ssaplash/notebooks/note_count_measure.py가 없음. 올리고 다시 실행"
if not os.path.isdir(INPUT_DIR) and os.path.isfile(INPUT_ZIP):
    zipfile.ZipFile(INPUT_ZIP).extractall(f"{WORK}/upload")
sets = {}
if os.path.isdir(INPUT_DIR):
    sets = {d: len(os.listdir(f"{INPUT_DIR}/{d}")) for d in sorted(os.listdir(INPUT_DIR))}
print("입력 파일 수:", sets)
assert sets, "입력이 없음. ssaplash/upload/note_count_inputs.zip을 올리고 다시 실행"

running = subprocess.run(["pgrep", "-u", getpass.getuser(), "-f", "note_count_measure.py"],
                         capture_output=True, text=True).stdout.split()
if os.path.isfile(f"{OUT}/summary.md"):
    print("이미 끝남. 2번 셀로 결과를 봄 (다시 측정하려면 results/note-count 폴더 이름을 바꾸고 이 셀을 다시 실행)")
elif running:
    print("이미 도는 중. pid", running, "→ 2번 셀로 진행을 봄")
else:
    os.makedirs(OUT, exist_ok=True)
    cmd = ["taskset", "-c", "0-127", f"{AI}/.venv/bin/python", SCRIPT,
           "--inputs", INPUT_DIR, "--out", OUT, "--device", "cuda"]
    proc = subprocess.Popen(["nohup", *cmd], cwd=AI, env=env, stdout=open(LOG, "a"),
                            stderr=subprocess.STDOUT, start_new_session=True)
    print("시작, pid", proc.pid, "로그", LOG)
# 확인:
#  - GPU 한 줄(index 6, 남은 메모리 10GB 이상), load 한 줄, 운영 워커 코드 커밋 한 줄, '채보 모델: 이름' 한 줄
#  - '입력 파일 수: {'extra': 9, 'musdb_mix': 50, 'musdb_nodrum': 50, 'speech': 10, 'talk_bgm': 10}'
#  - 마지막 줄이 '시작, pid 숫자 로그 .../results/note_count.log'. 다시 실행하면 '이미 도는 중' 또는 '이미 끝남'
#  - AssertionError가 나면 메시지에 적힌 파일을 올리고 이 셀을 다시 실행

In [ ]:
# 2. 진행 확인·결과 (여러 번 쳐도 됨. 커널이 다시 시작됐으면 1번을 먼저 실행)
!tail -4 $LOG | cut -c1-220
!pgrep -af note_count_measure.py | cut -c1-120 || echo "측정 프로세스 없음"
rows = f"{OUT}/rows.csv"
if os.path.isfile(rows):
    print("끝난 파일", sum(1 for _ in open(rows, encoding="utf-8")) - 1, "개 / 전체", sum(sets.values()), "개")
if os.path.isfile(f"{OUT}/summary.md"):
    print(open(f"{OUT}/summary.md", encoding="utf-8").read())
    zpath = f"{WORK}/results/note_count.zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        for f in (rows, f"{OUT}/summary.md", LOG):
            z.write(f, f"note-count/{os.path.basename(f)}")
    print("묶음", zpath, os.path.getsize(zpath) // 1024, "KB")
!cd $AI && git log --oneline -1
# 확인:
#  - 도는 중이면 로그 끝에 '[note-count] 시각 37/129 musdb_nodrum/12_이름.mp3: 노트 5개, 1분당 3.3' 같은 줄이 늘어남
#    처음에는 모델 올리는 줄 3~4개가 먼저 찍힘. 파일 하나에 10~20초
#  - '음악 여부 검사는 측정하지 않음: ...' 줄이 나오면 뒤에 적힌 사유를 확인함. music-check extra나 분류 모델 가중치가 없는 경우임
#  - 끝나면 '# 1분당 노트 수 측정 요약' 표 두 개와 '묶음 .../results/note_count.zip 숫자 KB' 줄이 찍힘
#    이 zip을 내려받아 채팅에 올리거나, 출력된 표와 마지막 커밋 줄을 그대로 붙여 줌
#  - '측정 프로세스 없음'인데 표가 없으면 멈춘 것. 새 셀에서 아래 한 줄을 치고 나온 내용을 그대로 붙여 준 뒤 1번 셀을 다시 실행:
#    !tail -30 ~/ssaplash/results/note_count.log